# Deep Learning Laboratory: 10 Comprehensive Experiments
Complete lab suite covering Foundational Neural Networks, Regularization, Optimization, CNNs, LSTMs, Sequence Modeling, Seq2Seq, RBMs, Autoencoders, and Generative Adversarial Networks (GANs).

## Experiment 1: Multi-Layer Perceptron (MLP) for XOR Logic Gate
**Aim:** Train a Multi-Layer Perceptron to solve the non-linearly separable XOR problem using binary cross-entropy and Adam optimizer.

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

model = Sequential([
    Input(shape=(2,)),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=1000, verbose=0)

loss, acc = model.evaluate(X, y, verbose=0)
print(f"Accuracy: {acc * 100:.2f}%")
preds = (model.predict(X) > 0.5).astype(int)
for i in range(len(X)):
    print(f"Input: {X[i]} -> Predicted: {preds[i][0]} (Actual: {y[i]})")

## Experiment 2: Regularization Techniques & Data Augmentation on MNIST
**Aim:** Prevent overfitting using L1/L2 weight penalties, Dropout layers, Gaussian noise, and ImageDataGenerator on MNIST.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
y_train = tf.keras.utils.to_categorical(y_train, 10)

noise = 0.05 * np.random.normal(size=x_train.shape)
x_train = np.clip(x_train + noise, 0.0, 1.0)

datagen = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1, validation_split=0.2)

model = Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4)),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(datagen.flow(x_train, y_train, batch_size=128, subset='training'),
          validation_data=datagen.flow(x_train, y_train, batch_size=128, subset='validation'),
          epochs=5)

## Experiment 3: Comparison of Optimization Algorithms (SGD vs Momentum vs Adam)
**Aim:** Train identical networks with SGD, SGD + Momentum, and Adam to observe loss convergence curves.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Sequential, optimizers
import matplotlib.pyplot as plt

# 1. XOR Dataset
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float32)
y = np.array([[0],[1],[1],[0]], dtype=np.float32)

def train_with_optimizer(opt):
    model = Sequential([
        layers.Dense(4, input_dim=2, activation='tanh'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=opt, loss='binary_crossentropy')
    history = model.fit(X, y, epochs=500, verbose=0)
    return history.history['loss']

# 2. Train with each optimizer and compare
sgd_losses = train_with_optimizer(optimizers.SGD(learning_rate=0.1))
momentum_losses = train_with_optimizer(optimizers.SGD(learning_rate=0.1, momentum=0.9))
adam_losses = train_with_optimizer(optimizers.Adam(learning_rate=0.1))

print("Final Loss (SGD):", sgd_losses[-1])
print("Final Loss (Momentum):", momentum_losses[-1])
print("Final Loss (Adam):", adam_losses[-1])

# 3. Plotting the losses
plt.figure(figsize=(10, 6))
plt.plot(sgd_losses, label='SGD')
plt.plot(momentum_losses, label='Momentum')
plt.plot(adam_losses, label='Adam')
plt.title('Loss during Training for Different Optimizers (XOR Problem)')
plt.xlabel('Epochs')
plt.ylabel('Binary Crossentropy Loss')
plt.legend()
plt.grid(True)
plt.show()

## Experiment 4: Convolutional Neural Network (CNN) for Image Classification
**Aim:** Classify handwritten digits from MNIST using a 2D CNN with Conv2D, MaxPooling2D, Flatten, and Dense layers.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

# 1. Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# 2. CNN Model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# 3. Compile, Train & Evaluate
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=3, batch_size=64, validation_split=0.1)

loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

# 4. Plotting training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Experiment 5: Long Short-Term Memory (LSTM) with Word Embeddings for Sentiment Analysis
**Aim:** Train an LSTM network with embedding layers to classify sentiment in IMDB movie reviews.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import matplotlib.pyplot as plt

vocab_size = 10000
maxlen = 200
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
x_train = pad_sequences(x_train, maxlen=maxlen, padding='post')
x_test = pad_sequences(x_test, maxlen=maxlen, padding='post')

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=3, batch_size=64, validation_split=0.2)
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Experiment 6: Sequence Modeling: Simple RNN vs. Bidirectional RNN
**Aim:** Compare standard unidirectional Simple RNN and Bidirectional RNN models on synthetic numerical sequence data.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# 1. Synthetic Sequence Dataset
seq_len = 10
X, y = [], []
for _ in range(1000):
    s = np.random.randint(0, 100)
    seq = np.arange(s, s + seq_len)
    X.append(seq[:-1])
    y.append(seq[1:])
X = np.array(X, dtype=np.float32)[..., np.newaxis]
y = np.array(y, dtype=np.float32)[..., np.newaxis]

# 2. Simple RNN
rnn = Sequential([
    layers.SimpleRNN(32, return_sequences=True, input_shape=(seq_len - 1, 1)),
    layers.Dense(1)
])
rnn.compile(optimizer='adam', loss='mse')
h1 = rnn.fit(X, y, epochs=15, verbose=0)

# 3. Bidirectional RNN
birnn = Sequential([
    layers.Bidirectional(layers.SimpleRNN(32, return_sequences=True), input_shape=(seq_len - 1, 1)),
    layers.Dense(1)
])
birnn.compile(optimizer='adam', loss='mse')
h2 = birnn.fit(X, y, epochs=15, verbose=0)

print(f"Final Loss (Standard RNN): {h1.history['loss'][-1]:.4f}")
print(f"Final Loss (BiRNN): {h2.history['loss'][-1]:.4f}")

## Experiment 7: Sequence-to-Sequence (Seq2Seq) Encoder-Decoder with LSTM
**Aim:** Implement an Encoder-Decoder recurrent architecture using LSTM hidden states for sequence transformation.

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# 1. Generate Synthetic Reversed Sequences
num_samples, timesteps, features = 1000, 5, 1
X = np.random.rand(num_samples, timesteps, features)
Y = np.flip(X, axis=1)

# 2. Encoder
enc_in = Input(shape=(timesteps, features))
_, state_h, state_c = LSTM(32, return_state=True)(enc_in)

# 3. Decoder
dec_in = Input(shape=(timesteps, features))
dec_lstm = LSTM(32, return_sequences=True)(dec_in, initial_state=[state_h, state_c])
dec_out = Dense(features)(dec_lstm)

# 4. Compile & Train
model = Model([enc_in, dec_in], dec_out)
model.compile(optimizer='adam', loss='mse')
model.fit([X, Y], Y, epochs=10, batch_size=32, verbose=0)

# 5. Test
pred = model.predict([X[:1], Y[:1]], verbose=0)
print("Input:    ", X[0].squeeze())
print("Predicted:", pred[0].squeeze())
print("Actual:   ", Y[0].squeeze())

## Experiment 8: Restricted Boltzmann Machine (RBM) with Contrastive Divergence
**Aim:** Train an energy-based generative model using Gibbs sampling and Contrastive Divergence (CD-1) on binarized MNIST.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 1. Load & Binarize MNIST
(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = (x_train.reshape(-1, 784).astype('float32') / 255.0 > 0.5).astype('float32')
dataset = tf.data.Dataset.from_tensor_slices(x_train).batch(64)

# 2. RBM Parameters
v_dim, h_dim = 784, 128
W = tf.Variable(tf.random.normal([v_dim, h_dim], stddev=0.01))
v_bias = tf.Variable(tf.zeros([v_dim]))
h_bias = tf.Variable(tf.zeros([h_dim]))

def sample(p):
    return tf.nn.relu(tf.sign(p - tf.random.uniform(tf.shape(p))))

# 3. Contrastive Divergence Training
lr = 0.05
epoch_losses = []
for epoch in range(3):
    total_loss = 0.0
    for v in dataset:
        # Positive phase
        h_prob = tf.sigmoid(tf.matmul(v, W) + h_bias)
        h0 = sample(h_prob)

        # Negative phase
        v1 = sample(tf.sigmoid(tf.matmul(h0, tf.transpose(W)) + v_bias))
        h1_prob = tf.sigmoid(tf.matmul(v1, W) + h_bias)

        # Updates
        bs = tf.cast(tf.shape(v)[0], tf.float32)
        W.assign_add(lr * (tf.matmul(tf.transpose(v), h_prob) - tf.matmul(tf.transpose(v1), h1_prob)) / bs)
        v_bias.assign_add(lr * tf.reduce_mean(v - v1, axis=0))
        h_bias.assign_add(lr * tf.reduce_mean(h_prob - h1_prob, axis=0))

        total_loss += tf.reduce_mean(tf.square(v - v1)).numpy()
    epoch_avg_loss = total_loss / len(dataset)
    epoch_losses.append(epoch_avg_loss)
    print(f"Epoch {epoch+1} Loss: {epoch_avg_loss:.4f}")

# 4. Plotting the MSE Loss
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o')
plt.title('RBM Training MSE Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.show()

## Experiment 9: Denoising Autoencoder for Image Reconstruction
**Aim:** Train an autoencoder with convolutional layers (and dense bottleneck) to remove additive Gaussian noise from MNIST images.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# 1. Load data and scale pixels between 0 and 1
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.reshape(-1, 28, 28, 1) / 255.0

# 2. Add random noise to images
noise_train = np.random.normal(0, 0.3, x_train.shape)
noise_test = np.random.normal(0, 0.3, x_test.shape)
x_train_noisy = np.clip(x_train + noise_train, 0.0, 1.0)
x_test_noisy = np.clip(x_test + noise_test, 0.0, 1.0)

# 3. Build model (shrinks image to clean it, then expands it back)
model = models.Sequential([
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.MaxPooling2D(2, padding='same'),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.UpSampling2D(2),
    layers.Conv2D(1, 3, activation='sigmoid', padding='same')
])

# 4. Train the model to turn noisy images into clean images
model.compile(optimizer='adam', loss='binary_crossentropy')
model.fit(x_train_noisy, x_train, epochs=5, batch_size=256, validation_data=(x_test_noisy, x_test))

# 5. Predict and display 10 samples (Row 1: Clean, Row 2: Noisy, Row 3: Fixed)
clean_preds = model.predict(x_test_noisy[:10])

plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(3, 10, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.subplot(3, 10, 10 + i + 1)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.subplot(3, 10, 20 + i + 1)
    plt.imshow(clean_preds[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.show()

## Experiment 10: Generative Adversarial Network (GAN) for Synthetic Digit Generation
**Aim:** Train a Deep Convolutional GAN (Generator and Discriminator) in a minimax game to synthesize novel handwritten digits from Gaussian noise.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype('float32') - 127.5) / 127.5
x_train = x_train.reshape(-1, 28, 28, 1)

generator = tf.keras.Sequential([
    layers.Dense(7 * 7 * 64, input_dim=100),
    layers.Reshape((7, 7, 64)),
    layers.Conv2DTranspose(32, 5, strides=2, padding="same", activation="relu"),
    layers.Conv2DTranspose(1, 5, strides=2, padding="same", activation="tanh")
])

discriminator = tf.keras.Sequential([
    layers.Conv2D(32, 5, strides=2, padding="same", input_shape=(28, 28, 1)),
    layers.LeakyReLU(0.2),
    layers.Flatten(),
    layers.Dense(1, activation="sigmoid")
])
discriminator.compile(optimizer='adam', loss='binary_crossentropy')

discriminator.trainable = False
gan = tf.keras.Sequential([generator, discriminator])
gan.compile(optimizer='adam', loss='binary_crossentropy')

test_noise = np.random.normal(0, 1, (1, 100))

for step in range(1501):
    noise = np.random.normal(0, 1, (128, 100))
    fakes = generator.predict(noise, verbose=0)
    reals = x_train[np.random.randint(0, len(x_train), 128)]

    d_loss = discriminator.train_on_batch(
        np.concatenate([reals, fakes]),
        np.concatenate([np.ones((128, 1)), np.zeros((128, 1))])
    )

    g_loss = gan.train_on_batch(noise, np.ones((128, 1)))

    if step % 500 == 0:
        print(f"Step {step} | D loss: {d_loss:.3f} | G loss: {g_loss:.3f}")
        img = generator.predict(test_noise, verbose=0)[0, :, :, 0]
        plt.imshow((img + 1) / 2, cmap='gray')
        plt.axis('off')
        plt.show()